In [1]:
import numpy as np
import pandas as pd
import h5py

from Account import *
# from Agent import *
from DataAsset import * 
from Exchnage import *
from Updater import *

In [2]:
import numpy as np


class Agent:
    def __init__(self, initial_cash, 출력=True):
        self._출력 = 출력
        self._date = None

        self._initial_cash = initial_cash
        self.cash = initial_cash
        self.total_balance = initial_cash

        self.accounts = dict()  # 여러 자산군에 대한 계좌
        self._basket = dict()  # 해당일에 구매리스트를 받기위한 바구니

        self.report = {"수익률(%)": [], "누적수익률(%)": [], "총자산(원)": [],
                       "CAGR(%)": [], "일평균수익률(%)": [], "MDD": [], "최대수익률(%)": [], "날짜": []}

    def set_account(self, name, account):
        self.accounts[name] = account  # 계좌등록
        # 해당 자산에 대해서 바스켓 생성
        self._basket[name] = {"구매이름": [], "구매가격": [], "구매수량": [], "구매종류": [], "구매시간": [],
                              "판매이름": [], "판매가격": [], "판매수량": [], "판매종류": [], "판매시간": []}

    def buy(self, name_account, name, 주문가격, 주문수량, 주문종류="limit", 주문시간="장중"):
        self._basket[name_account]["구매이름"].append(name)
        self._basket[name_account]["구매가격"].append(주문가격)
        self._basket[name_account]["구매수량"].append(주문수량)
        self._basket[name_account]["구매종류"].append(주문종류)
        self._basket[name_account]["구매시간"].append(주문시간)

    def sell(self, name_account, name, 주문가격, 주문수량, 주문종류="limit", 주문시간="장중"):
        self._basket[name_account]["판매이름"].append(name)
        self._basket[name_account]["판매가격"].append(주문가격)
        self._basket[name_account]["판매수량"].append(주문수량)
        self._basket[name_account]["판매종류"].append(주문종류)
        self._basket[name_account]["판매시간"].append(주문시간)

    def _reset_basket(self, name):
        self._basket[name] = {"구매이름": [], "구매가격": [], "구매수량": [], "구매종류": [], "구매시간": [],
                              "판매이름": [], "판매가격": [], "판매수량": [], "판매종류": [], "판매시간": []}

    def _shopping_basket(self):
        for name_account in self.accounts.keys():
            self._sell_list(name_account, self._basket[name_account]["판매이름"],
                      self._basket[name_account]["판매가격"], self._basket[name_account]["판매수량"],
                      주문종류=self._basket[name_account]["판매종류"], 주문시간=self._basket[name_account]["판매시간"])

            self._buy_list(name_account, self._basket[name_account]["구매이름"],
                     self._basket[name_account]["구매가격"], self._basket[name_account]["구매수량"],
                     주문종류=self._basket[name_account]["구매종류"], 주문시간=self._basket[name_account]["구매시간"])

            self._reset_basket(name_account)

    def _buy_list(self, name_account, names, 주문가격, 주문수량, 주문종류=None, 주문시간=None):
        주문가격 = np.array(주문가격)
        주문수량 = np.array(주문수량)

        cash_required = np.sum(주문가격 * 주문수량)
        if cash_required > self.cash:
            idx = np.sum(np.cumsum(주문가격 * 주문수량) < cash_required) - 1
            if idx == -1:
                return False

            names = names[:idx]
            주문가격 = 주문가격[:idx]
            주문수량 = 주문수량[:idx]

            if 주문종류 is not None:
                주문종류 = 주문종류[:idx]
            if 주문시간 is not None:
                주문시간 = 주문시간[:idx]

        cash_consumed = self.accounts[name_account].buy(names, 주문가격, 주문수량, 주문종류=주문종류, 주문시간=주문시간)
        self.cash -= cash_consumed

        return True

    def _sell_list(self, name_account, names, 주문가격, 주문수량, **kwds):
        주문가격 = np.array(주문가격)
        주문수량 = np.array(주문수량)

        cash_earned = self.accounts[name_account].sell(names, 주문가격, 주문수량, **kwds)
        self.cash += cash_earned

    def update_date(self, date):
        self._shopping_basket()
        self.total_balance = 0

        for key in self.accounts.keys():
            self.accounts[key].update_from_agent(date)
            self.total_balance += self.accounts[key].get_total_balance()

        self.total_balance += self.cash
        self._update_report()

        self._date = date

    def init(self, date):
        self._date = date
        self.report = {"수익률(%)": [0.0], "누적수익률(%)": [0.0], "총자산(원)": [self._initial_cash],
                       "CAGR(%)": [0.0], "일평균수익률(%)": [0.0], "MDD": [0.0], "최대수익률(%)": [0], "날짜": [self._date]}

    def _update_report(self):
        당일수익률 = (self.total_balance - self.report["총자산(원)"][-1]) / self.report["총자산(원)"][-1] * 100
        누적수익률 = (self.total_balance / self._initial_cash - 1) * 100
        총자산 = self.total_balance
        경과 = (self._date - self.report["날짜"][0]).days + 1

        일평균수익률 = ((누적수익률 / 100 + 1) ** (1 / 경과) - 1) * 100
        CAGR = ((일평균수익률 / 100 + 1) ** 365 - 1) * 100

        최대수익률 = max(누적수익률, np.max(self.report["누적수익률(%)"]))
        DD = (누적수익률 - 최대수익률) / (최대수익률 + 100) * 100
        MDD = min(DD, np.min(self.report["MDD"]))

        레포트_당일 = {"수익률(%)": 당일수익률, "누적수익률(%)": 누적수익률, "총자산(원)": 총자산,
                  "CAGR(%)": CAGR, "일평균수익률(%)": 일평균수익률, "MDD": MDD, "최대수익률(%)": 최대수익률, "날짜": self._date}

        for field in self.report.keys():
            self.report[field].append(레포트_당일[field])

        if self._출력:
            print("장마감 : ", self._date, "\n\n",
                  "당일수익률(%) : ", 당일수익률, "\n",
                  "누적수익률(%) : ", 누적수익률, "\n",
                  "CAGR(%)", CAGR, "\n",
                  "MDD : ", MDD, "\n",
                  "총자산(원) : ", 총자산, "\n",
                  "--------------------------------------------------\n")


# with pandas dataframe

In [3]:
%%time
f = h5py.File("./data/test.hdf5", "r")
array_stock, axis_stock = load_data(f, "stock", chunks=10, in_memory=False)
array_value, axis_value = load_data(f, "value", chunks=10, in_memory=False)

data_stock = DataAsset(array_stock, axis_stock)
data_value = DataAsset(array_value, axis_value)

updater = Updater(pd.Timestamp(2018, 1, 1), data_stock.dates)

# 거래소 생성
exchange_stock = Exchange()
exchange_stock.set_DataAsset(data_stock)

# 주식 계좌 생성
stock_account = StockAccount(exchange_stock, 출력=False)

# 거래 에이전트 생성 및 주식 계좌 등록
agent = Agent(1e8, 출력=False)
agent.set_account("stock", stock_account)

# 날짜가 변할시 업데이트 요청
updater.set_data(data_stock)
updater.set_data(data_value)

updater.set_agent(agent)
updater.set_exchange(exchange_stock)

updater.initialization()
columns = ["상장시가총액(원)", "지배주주순이익(원)(직전4분기)", "지배주주지분(원)",
           "현금흐름(원)(직전4분기)", "매출액(원)(직전4분기)"]

while updater._date != updater._list_date[-1]:
    print(updater._date)
    fin_stat = data_value.get_info(updater._date, num=2,
                                   fields=columns)

#     df = pd.DataFrame(fin_stat[-2], index=data_value.codes, columns=columns)
#     df = df[~np.isnan(df["상장시가총액(원)"])]  # 상장종목 고려
#     df = df.sort_values(by=['상장시가총액(원)']).iloc[:int(len(df.index) * 0.3)]  # 소형주

#     # 종목선정
#     df["PER"] = df["상장시가총액(원)"] / df["지배주주순이익(원)(직전4분기)"]
#     df["PBR"] = df["상장시가총액(원)"] / df["지배주주지분(원)"]
#     df["PCR"] = df["상장시가총액(원)"] / df["현금흐름(원)(직전4분기)"]
#     df["PSR"] = df["상장시가총액(원)"] / df["매출액(원)(직전4분기)"]

#     df = df[df["PER"] > 0]
#     df = df[df["PBR"] > 0]
#     df = df[df["PCR"] > 0]
#     df = df[df["PSR"] > 0]

#     df["Rank"] = (df["PER"].rank() + df["PBR"].rank() + df["PCR"].rank() + df["PSR"].rank()).rank()
#     df = df[df["Rank"] < 51]

#     매수종목 = np.sort(df.index)

    # 매도
#     매도종목 = list(agent.accounts["stock"].keys())
#     현재가 = data_stock.get_info(updater._date, codes=매도종목, fields=["현재가"]).reshape(-1)
#     매도수량 = [agent.accounts["stock"][종목코드]["보유수량"] for 종목코드 in 매도종목]
    
#     for i in range(len(매도종목)):
#         agent.sell("stock", 매도종목[i], 현재가[i], 매도수량[i])

#     # 매수
#     현재가 = data_stock.get_info(updater._date, codes=매수종목, fields=["현재가"]).reshape(-1).astype("f")
#     거래가능 = ~np.isnan(현재가)
#     매수수량 = (agent.cash / 50 / 현재가[거래가능]).astype("i")
    
#     for i in range(len(np.where(거래가능)[0])):
#         agent.buy("stock", 매수종목[거래가능][i], 현재가[거래가능][i], 매수수량[i])

    updater.update()

2018-01-02 00:00:00
2018-01-03 00:00:00
2018-01-04 00:00:00
2018-01-05 00:00:00
2018-01-08 00:00:00
2018-01-09 00:00:00
2018-01-10 00:00:00
2018-01-11 00:00:00
2018-01-12 00:00:00
2018-01-15 00:00:00
2018-01-16 00:00:00
2018-01-17 00:00:00
2018-01-18 00:00:00
2018-01-19 00:00:00
2018-01-22 00:00:00
2018-01-23 00:00:00
2018-01-24 00:00:00
2018-01-25 00:00:00
2018-01-26 00:00:00


F:\Dropbox\site-packages\systrading\backtest\Toos\SushiLife\SushiLife\Exchnage.py:77: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  cond2 = (주문시간 == "장전") & (OCLHVVM[:, 0] >= 주문가격)  # 체결, 장 시작과 동시에 체결
F:\Dropbox\site-packages\systrading\backtest\Toos\SushiLife\SushiLife\Exchnage.py:80: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  cond5 = (주문종류 == "조건부지정가")  # 체결, 장 마감시 체결
F:\Dropbox\site-packages\systrading\backtest\Toos\SushiLife\SushiLife\Exchnage.py:36: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  cond2 = (주문시간 == "장전") & (OCLHVVM[:, 0] <= 주문가격)  # 체결, 장 시작과 동시에 체결
F:\Dropbox\site-packages\systrading\backtest\Toos\SushiLife\SushiLife\Exchnage.py:39: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future w

2018-01-29 00:00:00
2018-01-30 00:00:00
2018-01-31 00:00:00
2018-02-01 00:00:00
2018-02-02 00:00:00
2018-02-05 00:00:00
2018-02-06 00:00:00
2018-02-07 00:00:00
2018-02-08 00:00:00
2018-02-09 00:00:00
2018-02-12 00:00:00
2018-02-13 00:00:00
2018-02-14 00:00:00
2018-02-19 00:00:00
2018-02-20 00:00:00
2018-02-21 00:00:00
2018-02-22 00:00:00
2018-02-23 00:00:00
2018-02-26 00:00:00
2018-02-27 00:00:00
2018-02-28 00:00:00
2018-03-02 00:00:00
2018-03-05 00:00:00
2018-03-06 00:00:00
2018-03-07 00:00:00
2018-03-08 00:00:00
2018-03-09 00:00:00
2018-03-12 00:00:00
2018-03-13 00:00:00
2018-03-14 00:00:00
2018-03-15 00:00:00
2018-03-16 00:00:00
2018-03-19 00:00:00
2018-03-20 00:00:00
2018-03-21 00:00:00
2018-03-22 00:00:00
2018-03-23 00:00:00
2018-03-26 00:00:00
2018-03-27 00:00:00
2018-03-28 00:00:00
2018-03-29 00:00:00
2018-03-30 00:00:00
2018-04-02 00:00:00
2018-04-03 00:00:00
2018-04-04 00:00:00
2018-04-05 00:00:00
2018-04-06 00:00:00
2018-04-09 00:00:00
2018-04-10 00:00:00
2018-04-11 00:00:00


In [4]:
%%time
f = h5py.File("./data/test.hdf5", "r")
array_stock, axis_stock = load_data(f, "stock", chunks=5, in_memory=False)
array_value, axis_value = load_data(f, "value", chunks=5, in_memory=False)

data_stock = DataAsset(array_stock, axis_stock, chunks=5)
data_value = DataAsset(array_value, axis_value, chunks=5)

updater = Updater(pd.Timestamp(2018, 1, 1), data_stock.dates)

# 거래소 생성
exchange_stock = Exchange()
exchange_stock.set_DataAsset(data_stock)

# 주식 계좌 생성
stock_account = StockAccount(exchange_stock, 출력=False)

# 거래 에이전트 생성 및 주식 계좌 등록
agent = Agent(1e8, 출력=False)
agent.set_account("stock", stock_account)

# 날짜가 변할시 업데이트 요청
updater.set_data(data_stock)
updater.set_data(data_value)

updater.set_agent(agent)
updater.set_exchange(exchange_stock)

updater.initialization()
columns = ["상장시가총액(원)", "지배주주순이익(원)(직전4분기)", "지배주주지분(원)",
           "현금흐름(원)(직전4분기)", "매출액(원)(직전4분기)"]

while updater._date != updater._list_date[-1]:
    print(updater._date)
    fin_stat = data_value.get_info(updater._date, num=2,
                                   fields=columns)

    df = pd.DataFrame(fin_stat[-2], index=data_value.codes, columns=columns)
    df = df[~np.isnan(df["상장시가총액(원)"])]  # 상장종목 고려
    df = df.sort_values(by=['상장시가총액(원)']).iloc[:int(len(df.index) * 0.3)]  # 소형주

    # 종목선정
    df["PER"] = df["상장시가총액(원)"] / df["지배주주순이익(원)(직전4분기)"]
    df["PBR"] = df["상장시가총액(원)"] / df["지배주주지분(원)"]
    df["PCR"] = df["상장시가총액(원)"] / df["현금흐름(원)(직전4분기)"]
    df["PSR"] = df["상장시가총액(원)"] / df["매출액(원)(직전4분기)"]

    df = df[df["PER"] > 0]
    df = df[df["PBR"] > 0]
    df = df[df["PCR"] > 0]
    df = df[df["PSR"] > 0]

    df["Rank"] = (df["PER"].rank() + df["PBR"].rank() + df["PCR"].rank() + df["PSR"].rank()).rank()

    df = df[df["Rank"] < 51]

    매수종목 = np.sort(df.index)

    # 매도
    매도종목 = list(agent.accounts["stock"].keys())
    현재가 = data_stock.get_info(updater._date, codes=매도종목, fields=["현재가"]).reshape(-1)
    매도수량 = [agent.accounts["stock"][종목코드]["보유수량"] for 종목코드 in 매도종목]
    
    for i in range(len(매도종목)):
        agent.sell("stock", 매도종목[i], 현재가[i], 매도수량[i])

    # 매수
    현재가 = data_stock.get_info(updater._date, codes=매수종목, fields=["현재가"]).reshape(-1).astype("f")
    거래가능 = ~np.isnan(현재가)
    매수수량 = (agent.cash / 50 / 현재가[거래가능]).astype("i")
    
    for i in range(len(np.where(거래가능)[0])):
        agent.buy("stock", 매수종목[거래가능][i], 현재가[거래가능][i], 매수수량[i])

    updater.update()

TypeError: __init__() got an unexpected keyword argument 'chunks'